# Black-Litterman Asset Allocation

### Wealth Management Portfolio Optimization — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), and basic understanding of wealth management and portfolio optimisation terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to wealth management and portfolio optimisation.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Portfolio optimisation, asset allocation, risk management, and investment strategy for wealth management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Pure Markowitz optimization often produces extreme, unstable results (e.g., "put 100% in Gold"). The Black-Litterman model fixes this by starting with the market's consensus and only tilting the portfolio where the investor has strong opinions.

**Input data used:** Market capitalizations (for weights), covariance matrix, and investor views.

**What we calculate:** 'Posterior' expected returns and updated asset weights.

**Math method used:** Bayesian Inference (combining a prior with new evidence).

**Learning goal:** Understand how to blend data-driven market consensus with expert intuition.

**Primary stakeholders:** wealth advisors, portfolio managers, investment committee members, and financial planners.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Define Market Equilibrium

We start with the current "Market Portfolio" (how much of the total world wealth is in each asset) and use it to work backwards to find out what returns the market is currently pricing in.

In [ ]:
assets = ['Stocks', 'Bonds', 'Real Estate', 'Gold']
n_assets = len(assets)

# Market Weights (Capitalization weighted)
w_market = np.array([0.45, 0.40, 0.10, 0.05])

# Covariance Matrix (from previous notebook/historical data)
sigma = np.array([
    [0.0324, 0.0009, 0.0130, 0.0000],
    [0.0009, 0.0025, 0.0012, 0.0022],
    [0.0130, 0.0012, 0.0144, 0.0018],
    [0.0000, 0.0022, 0.0018, 0.0225]
])

risk_aversion = 2.5 # Lambda

# Implied Equilibrium Excess Returns (Pi)
pi = risk_aversion * (sigma @ w_market)

df_mkt = pd.DataFrame({'Market Weight': w_market, 'Implied Return': pi}, index=assets)
print("Market Equilibrium Baseline:")
print(df_mkt)

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

An investor might have specific views. Let's say:
1. "Gold will return 4% absolute."
2. "Stocks will outperform Real Estate by 3%."

In [ ]:
# Q: The vector of views
Q = np.array([0.04, 0.03])

# P: The picking matrix (identifies which assets the views apply to)
# Row 1: Gold (index 3) = 4%
# Row 2: Stocks (0) - Real Estate (2) = 3%
P = np.array([
    [0, 0, 0, 1],
    [1, 0, -1, 0]
])

tau = 0.05 # Uncertainty constant
Omega = np.diag(np.diag(P @ (tau * sigma) @ P.T)) # Uncertainty of views

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
# Intermediate terms
A = np.linalg.inv(tau * sigma)
B = P.T @ np.linalg.inv(Omega) @ P
C = P.T @ np.linalg.inv(Omega) @ Q

# Posterior Returns
er = np.linalg.inv(A + B) @ (A @ pi + C)

# Posterior Weights
w_posterior = np.linalg.inv(risk_aversion * sigma) @ er
w_posterior /= np.sum(w_posterior) # Normalize

df_results = pd.DataFrame({
    'Equilibrium Return': pi,
    'Posterior Return': er,
    'Equilibrium Weight': w_market,
    'Posterior Weight': w_posterior
}, index=assets)

print("Comparison of Results:")
print(df_results)

In [ ]:
df_results[['Equilibrium Weight', 'Posterior Weight']].plot(kind='bar', figsize=(10, 6))
plt.title('How Investor Views Shift Asset Allocation')
plt.ylabel('Weight (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "How Investor Views Shift Asset Allocation". The chart highlights key patterns that inform the modelling approach.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **Stability:** Notice that even though we had a view on Gold, the weight didn't jump to 100%. It just 'tilted' in that direction.
2. **Relative Views:** Because we thought Stocks would beat Real Estate, the model increased Stock weight and decreased Real Estate weight.
3. **Conclusion:** Black-Litterman is the industry standard for large investment banks because it results in 'well-behaved' portfolios that make sense to human managers.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end wealth management and portfolio optimisation workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Optimisation models may suggest portfolios unsuitable for clients with different risk tolerances or time horizons.
- Model assumptions about return distributions may not hold during market crises.
- Fiduciary duty requires that model outputs support, not replace, professional judgement.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in wealth management and portfolio optimisation?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.